**Handling Nulls**

Detecting Nulls — isNull() and isNotNull()

dropna() — Remove Rows with Nulls

fillna() — Replace Nulls with a Value

coalesce() — Return First Non-Null Value

coalesce() vs fillna()
fillna() — replace nulls with a fixed constant value. Simple and fast.

coalesce() — replace nulls with the value from another column. Use when the fallback is dynamic, not a constant.

In [4]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-9")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


**Task 1**

Create a DataFrame with at least 5 rows and intentional nulls across multiple columns. Count the number of nulls in each column using the when(isNull()).sum() pattern.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

schema = StructType([
    StructField("cust_id", StringType(), True),
    StructField("cust_Name", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("unit_price", DoubleType(), True)
])

data = [
    ("C001", "John",  "25240", 350.0),
    (None,   "Bob",   "34526", 430.0),
    ("C002", "Alice", "43526", 55.0),
    ("C004", "Robin", None,    70.0),
    ("C005", None,    "44353", 150.0)
]

df = spark.createDataFrame(data, schema)
df.select(
    [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]
).show()

+-------+---------+--------+----------+
|cust_id|cust_Name|order_id|unit_price|
+-------+---------+--------+----------+
|      1|        1|       1|         0|
+-------+---------+--------+----------+



**Task 2**

Using your DataFrame from Task 1, drop rows where any column is null. Then drop rows where only specific columns (customer_id and unit_price) are null. Compare the row counts.

In [10]:
cleaned_df1=df.dropna()
print(f"total row count after droping rows where any column is null:{cleaned_df1.count()}")
cleaned_df2=df.dropna(subset=['cust_id','unit_price'])
print(f"total row count after dropping rows where only specific column are null: {cleaned_df2.count()}")

total row count after droping rows where any column is null:2
total row count after dropping rows where only specific column are null: 4



**task3**

Using fillna() with a dictionary, fill nulls in your DataFrame: string columns with "Unknown", numeric columns with 0. Show the result — no nulls should remain.

In [15]:
df.fillna({
    "cust_id":"Unknown",
    "cust_name":"Unknown",
    "order_id":"Unknown",
    "unit_price":0
}).show()

+-------+---------+--------+----------+
|cust_id|cust_Name|order_id|unit_price|
+-------+---------+--------+----------+
|   C001|     John|   25240|     350.0|
|Unknown|      Bob|   34526|     430.0|
|   C002|    Alice|   43526|      55.0|
|   C004|    Robin| Unknown|      70.0|
|   C005|  Unknown|   44353|     150.0|
+-------+---------+--------+----------+



**Task 4**
    
Create a DataFrame with two address columns — billing_address and shipping_address — where some rows have one or both as null. Use F.coalesce() to create a delivery_address column that picks the first non-null address, falling back to "Address Unknown".

In [ ]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

# Schema
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("billing_address", StringType(), True),
    StructField("shipping_address", StringType(), True)
])

# Data
data = [
    ("C001", "Delhi", "Noida"),
    ("C002", None, "Mumbai"),
    ("C003", "Pune", None),
    ("C004", None, None),
    ("C005", "Bangalore", "Hyderabad")
]
df = spark.createDataFrame(data, schema)
result = df.withColumn(
    "delivery_address",
    F.coalesce(
        F.col("billing_address"),
        F.col("shipping_address"),
        F.lit("Address Unknown")
    )
)

result.show()

+-----------+---------------+----------------+----------------+
|customer_id|billing_address|shipping_address|delivery_address|
+-----------+---------------+----------------+----------------+
|       C001|          Delhi|           Noida|           Delhi|
|       C002|           NULL|          Mumbai|          Mumbai|
|       C003|           Pune|            NULL|            Pune|
|       C004|           NULL|            NULL| Address Unknown|
|       C005|      Bangalore|       Hyderabad|       Bangalore|
+-----------+---------------+----------------+----------------+

